# EEG Bandpass Filtering Demo

This notebook demonstrates loading EEG data for a single subject and run, extracting the raw signal, applying a bandpass filter (0.1-20 Hz), and plotting the results before and after filtering.

## Context
- Data from distractor decoding experiment
- EEG sampled at 512 Hz
- Bandpass filter: 0.1-20 Hz (as per preprocessing pipeline)

In [1]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import mne
from scipy.signal import butter, filtfilt


In [2]:
# Function to load EEG data from .gdf file
def load_eeg_data(gdf_path):
    """
    Load EEG data from a .gdf file.
    
    Parameters:
    gdf_path (str): Path to the .gdf file
    
    Returns:
    raw_data (np.ndarray): EEG signal matrix (n_samples x n_channels)
    sfreq (float): Sampling frequency
    ch_names (list): Channel names
    """
    raw = mne.io.read_raw_gdf(gdf_path, preload=True, verbose=False)
    raw_data = raw.get_data()  # Shape: (n_channels, n_samples)
    raw_data = raw_data.T  # Transpose to (n_samples, n_channels) for consistency
    sfreq = raw.info['sfreq']
    ch_names = raw.ch_names
    return raw_data, sfreq, ch_names

In [3]:
# Function to apply bandpass filter
def apply_bandpass_filter(data, sfreq, low_freq=0.1, high_freq=20.0, order=4):
    """
    Apply bandpass filter to EEG data.
    
    Parameters:
    data (np.ndarray): EEG signal (n_samples x n_channels)
    sfreq (float): Sampling frequency
    low_freq (float): Low cutoff frequency
    high_freq (float): High cutoff frequency
    order (int): Filter order
    
    Returns:
    filtered_data (np.ndarray): Filtered EEG signal
    """
    nyquist = sfreq / 2
    low = low_freq / nyquist
    high = high_freq / nyquist
    b, a = butter(order, [low, high], btype='band')
    
    # Apply filter to each channel
    filtered_data = np.zeros_like(data)
    for ch in range(data.shape[1]):
        filtered_data[:, ch] = filtfilt(b, a, data[:, ch])
    
    return filtered_data

In [ ]:
# Load EEG data for one subject and run
# Path structure: /Users/hililbby/Library/CloudStorage/Box-Box/CNBI/Attention_distraction/project_healthy/{subject}/{subject_date}/{subject_datetime_task}/{subject_datetime_task}.gdf
# Example: /Users/hililbby/Library/CloudStorage/Box-Box/CNBI/Attention_distraction/project_healthy/e38/e38_20260228/e38_20260228162653_stroop/e38_20260228162653_stroop.gdf
gdf_path = '/Users/hililbby/Library/CloudStorage/Box-Box/CNBI/Attention_distraction/project_healthy/e38/e38_20260228/e38_20260228162653_stroop/e38_20260228162653_stroop.gdf'

try:
    raw_data, sfreq, ch_names = load_eeg_data(gdf_path)
    print(f"Data shape: {raw_data.shape}")
    print(f"Sampling rate: {sfreq} Hz")
    print(f"Number of channels: {len(ch_names)}")
    print(f"Channel names (first 10): {ch_names[:10]}")
except FileNotFoundError:
    print("GDF file not found. Please update the gdf_path variable with a valid path.")
    # For demonstration, create synthetic data
    sfreq = 512
    n_samples = 5120  # 10 seconds
    n_channels = 64
    ch_names = [f'EEG{i:03d}' for i in range(1, n_channels+1)]
    np.random.seed(42)
    # Create synthetic EEG-like signal
    t = np.arange(n_samples) / sfreq
    raw_data = np.zeros((n_samples, n_channels))
    for ch in range(n_channels):
        # Mix of sine waves and noise
        raw_data[:, ch] = (np.sin(2*np.pi*10*t) + 0.5*np.sin(2*np.pi*1*t) + 
                          0.1*np.random.randn(n_samples))
    print("Using synthetic data for demonstration")
    print(f"Data shape: {raw_data.shape}")
    print(f"Sampling rate: {sfreq} Hz")

# Based on doc/file_contents.md: channels 1-64 are EEG, 65-66 are EOG, and 67 is Status.
n_eeg_channels = min(64, raw_data.shape[1])
eeg_channel_indices = np.arange(n_eeg_channels)
eeg_channel_names = [ch_names[idx] for idx in eeg_channel_indices]
raw_eeg_data = raw_data[:, eeg_channel_indices]
print(f"Using {len(eeg_channel_indices)} EEG channels for plotting and filtering")

GDF file not found. Please update the gdf_path variable with a valid path.
Using synthetic data for demonstration
Data shape: (5120, 64)
Sampling rate: 512 Hz


In [ ]:
# Apply bandpass filter to EEG channels only
filtered_data = apply_bandpass_filter(raw_eeg_data, sfreq, low_freq=0.1, high_freq=20.0)
print("Filtering applied successfully")

In [ ]:
# Plot raw and filtered EEG separately with vertical offsets so channels remain legible.
window_start_sec = 30.0
window_duration_sec = 3.0
start_sample = int(window_start_sec * sfreq)
stop_sample = int((window_start_sec + window_duration_sec) * sfreq)
if stop_sample > raw_eeg_data.shape[0]:
    raise ValueError(
        f"Requested window ends at {window_start_sec + window_duration_sec:.1f} s, but the recording is only {raw_eeg_data.shape[0] / sfreq:.1f} s long."
    )
time_window = slice(start_sample, stop_sample)
t = np.arange(start_sample, stop_sample) / sfreq

def plot_stacked_eeg(data, time_vector, channel_names, title, color):
    windowed_data = data[time_window, :]
    signal_range = np.percentile(windowed_data, 95, axis=0) - np.percentile(windowed_data, 5, axis=0)
    offset_step = np.nanmedian(signal_range) * 1.5
    if not np.isfinite(offset_step) or offset_step == 0:
        offset_step = 1.0

    offsets = np.arange(windowed_data.shape[1]) * offset_step
    fig_height = max(12, 0.22 * len(channel_names))
    fig, ax = plt.subplots(figsize=(16, fig_height))

    for ch_idx, ch_name in enumerate(channel_names):
        ax.plot(time_vector, windowed_data[:, ch_idx] + offsets[ch_idx], color=color, linewidth=0.8)

    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('EEG Channels (offset)')
    ax.set_yticks(offsets)
    ax.set_yticklabels(channel_names)
    ax.grid(True, axis='x', alpha=0.3)
    ax.set_xlim(time_vector[0], time_vector[-1])
    plt.tight_layout()
    plt.show()

plot_stacked_eeg(raw_eeg_data, t, eeg_channel_names, 'Raw EEG Signals (Channels 1-64, 30-33 s)', 'steelblue')
plot_stacked_eeg(filtered_data, t, eeg_channel_names, 'Filtered EEG Signals (0.1-20 Hz, Channels 1-64, 30-33 s)', 'firebrick')

## Summary

This notebook demonstrated:
1. Loading EEG data from a .gdf file using MNE
2. Applying a bandpass filter (0.1-20 Hz) using SciPy
3. Plotting the signal before and after filtering

The filter removes high-frequency noise (>20 Hz) and very low-frequency drift (<0.1 Hz), preserving the main EEG frequency bands.

For actual analysis, replace the placeholder path with a real .gdf file path.